# 13 — Source-Use Strategies, a Published Baseline, and a Corrected Selector

Four additions to the four-corpus study, on the same corpora, caps, splits and seeds as notebooks 10 to 12.

Source use. Notebook 10 tested only naive pooling of the capped source set with the buffer (augment), which is the crudest way to retain source data. Two better uses are added: finetune, which continues the fitted source model on the buffer (Random Forest grows 100 further trees on the buffer under warm_start and keeps the originals; LightGBM continues boosting from the source booster via init_model; the MLP continues SGD from the source weights), and iw_augment, which reweights source rows by a density ratio from a domain discriminator trained on a class-balanced source-versus-buffer sample before pooling. iw_augment runs at the reference seed to match the existing augment cells.

Published baseline. INSOMNIA (Andresini et al., 2021) adapts a NIDS under drift by combining uncertainty-driven queries with label estimation. Two comparators are added: al_iterative, in which the budget is spent over three rounds with the retrained model selecting the highest-entropy pool rows after a stratified random first round, and insomnia, an INSOMNIA-style approximation in which the frozen source model pseudo-labels pool rows it scores above 0.9 or below 0.1 (capped at 50,000) and the analyst budget is spent on its highest-entropy rows.

Selector. Notebook 12's ceiling-gated selector used a three-fold cross-validated retraining estimate that returns zero whenever a class has fewer than three buffer rows, which is common at 144 to 217 flows. selector_v2 adapts the fold count to the rarest class, falls back to a two-fold stratified estimate and then to resubstitution, and records which mode was used so that cells resting on a fallback can be separated in analysis.

All strategies record the full metric set including false-positive rate. Results append to fc_results_v4.csv with resume-skip on (seed, source, target, model, budget, strategy).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, gc, time, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

BASE   = '/content/drive/MyDrive/drift-conference'
CACHE  = f'{BASE}/data/nfv2/cache'
RESULT = f'{BASE}/results/fourcorpus'

CFG = dict(
    corpora       = ['nf2018v2', 'nfunswv2', 'nftonv2', 'nfbotv2'],
    seeds         = [42, 43, 44],
    ref_seed      = 42,
    test_size     = 0.30,
    train_cap     = 250_000,
    eval_cap      = 200_000,
    pool_cap      = 200_000,
    budgets       = [0.0001, 0.001, 0.01, 0.05, 0.10],
    src_budgets   = [0.0001, 0.001, 0.01],
    acq_budgets   = [0.0001, 0.001],
    rf_estimators = 300,
    ft_add_trees  = 100,
    ft_epochs     = 30,
    ece_bins      = 15,
    mlp_hidden    = (128, 64),
    mlp_max_iter  = 100,
    al_rounds     = 3,
    pseudo_conf   = 0.9,
    pseudo_cap    = 50_000,
)
MODELS = ['rf', 'lgbm', 'mlp']
V4_CSV = f'{RESULT}/fc_results_v4.csv'
COLS = ['seed', 'source', 'target', 'model', 'budget', 'strategy', 'n_train',
        'macro_f1', 'weighted_f1', 'mcc', 'auprc_macro', 'fp_rate', 'brier', 'ece',
        'mcc_best_thr', 'thr_best', 'buffer_est', 'cv_est', 'cv_mode', 'n_pseudo',
        'n_benign_buf', 'fit_s']
print(json.dumps({k: str(v) for k, v in CFG.items()}, indent=2))

In [ ]:
DATASETS = {tag: pd.read_parquet(f'{CACHE}/{tag}_prepared.parquet') for tag in CFG['corpora']}
FEATURES = [c for c in DATASETS['nf2018v2'].columns if c not in ('Label', 'Attack')]
for tag, d in DATASETS.items():
    assert [c for c in d.columns if c not in ('Label', 'Attack')] == FEATURES, tag
print('features:', len(FEATURES))

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import (f1_score, matthews_corrcoef, average_precision_score,
                              brier_score_loss, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

F32_SAFE = 1e37

def ece_score(y_true, p_pos, n_bins=15):
    conf = np.maximum(p_pos, 1 - p_pos)
    correct = ((p_pos >= 0.5).astype(int) == y_true).astype(float)
    bins = np.linspace(0.5, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(conf, bins) - 1, 0, n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return ece

def mcc_from_counts(tp, fp, fn, tn):
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    with np.errstate(invalid='ignore', divide='ignore'):
        out = np.where(den > 0, num / den, 0.0)
    return out

def best_threshold_mcc(y_true, p_pos, n_grid=199):
    """Max MCC over a quantile grid of thresholds; robust to degenerate score distributions."""
    y = np.asarray(y_true).astype(int); p = np.asarray(p_pos)
    thr = np.unique(np.quantile(p, np.linspace(0.001, 0.999, n_grid)))
    if len(thr) < 2:
        thr = np.array([thr[0]]) if len(thr) else np.array([0.5])
    P = y.sum(); N = len(y) - P
    tp = np.array([(y[p >= t]).sum() for t in thr], dtype=float)
    fp = np.array([(p >= t).sum() for t in thr], dtype=float) - tp
    fn = P - tp; tn = N - fp
    mccs = mcc_from_counts(tp, fp, fn, tn)
    i = int(np.argmax(mccs))
    return float(mccs[i]), float(thr[i])

def all_metrics(y_true, p_pos, ece_bins=CFG['ece_bins']):
    y_true = np.asarray(y_true); p_pos = np.asarray(p_pos)
    pred = (p_pos >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    ap_att = average_precision_score(y_true, p_pos)
    ap_ben = average_precision_score(1 - y_true, 1 - p_pos)
    mb, tb = best_threshold_mcc(y_true, p_pos)
    return dict(
        macro_f1=f1_score(y_true, pred, average='macro'),
        weighted_f1=f1_score(y_true, pred, average='weighted'),
        mcc=matthews_corrcoef(y_true, pred),
        auprc_macro=(ap_att + ap_ben) / 2,
        fp_rate=fp / (fp + tn) if (fp + tn) else np.nan,
        brier=brier_score_loss(y_true, p_pos),
        ece=ece_score(y_true, p_pos, ece_bins),
        mcc_best_thr=mb, thr_best=tb,
    )

def clean_X(df, features, medians=None):
    X = df[features].astype('float64')
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.mask(X.abs() > F32_SAFE, np.nan)
    if medians is None:
        medians = X.median()
    return X.fillna(medians), medians

def make_model(name, seed, n_rows, rf_estimators=CFG['rf_estimators'], mlp_hidden=CFG['mlp_hidden'], mlp_max_iter=CFG['mlp_max_iter']):
    if name == 'rf':
        return RandomForestClassifier(n_estimators=rf_estimators, n_jobs=-1, random_state=seed)
    if name == 'lgbm':
        return lgb.LGBMClassifier(n_estimators=rf_estimators, random_state=seed, n_jobs=-1, verbosity=-1)
    # early stopping needs a stratifiable validation split; disable on tiny buffers
    small = n_rows < 5000
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(hidden_layer_sizes=mlp_hidden, activation='relu', solver='adam',
                              batch_size=min(1024, max(8, n_rows // 4)),
                              max_iter=(200 if small else mlp_max_iter),
                              early_stopping=not small, validation_fraction=0.05,
                              n_iter_no_change=5, random_state=seed)),
    ])

def platt_fit(scores, y):
    y = np.asarray(y)
    if len(np.unique(y)) < 2:
        return None
    lr = LogisticRegression(max_iter=1000)
    lr.fit(np.asarray(scores).reshape(-1, 1), y)
    return lr

def platt_apply(lr, scores, y_buf):
    if lr is None:
        return np.full(len(scores), float(np.asarray(y_buf)[0]))
    return lr.predict_proba(np.asarray(scores).reshape(-1, 1))[:, 1]

def stratified_frac(df, frac, seed, min_per_group=1):
    """Per-family sample of round(n*frac) rows, floored at min_per_group; no groupby.apply."""
    rng_state = seed
    parts = []
    for fam, g in df.groupby('Attack', sort=True):
        n = min(len(g), max(min_per_group, int(round(len(g) * frac))))
        parts.append(g.sample(n=n, random_state=rng_state))
    return pd.concat(parts).sample(frac=1.0, random_state=seed).reset_index(drop=True)

def stratified_cap(df, cap, seed):
    if len(df) <= cap:
        return df.reset_index(drop=True)
    return stratified_frac(df, cap / len(df), seed)

In [ ]:
import numpy as np, pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import matthews_corrcoef

def mcc_from_counts(tp, fp, fn, tn):
    num = tp * tn - fp * fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    with np.errstate(invalid='ignore', divide='ignore'):
        return np.where(den > 0, num / den, 0.0)

def best_threshold_mcc(y_true, p_pos):
    y = np.asarray(y_true).astype(np.int64); p = np.asarray(p_pos, dtype=np.float64)
    order = np.argsort(-p, kind='mergesort'); ps, ys = p[order], y[order]
    P = int(ys.sum()); N = len(ys) - P
    tp = np.cumsum(ys); fp = np.cumsum(1 - ys)
    last = np.r_[ps[1:] != ps[:-1], True]
    tp, fp, cuts = tp[last].astype(float), fp[last].astype(float), ps[last]
    mccs = mcc_from_counts(tp, fp, P - tp, N - fp)
    i = int(np.argmax(mccs))
    return (0.0, float('inf')) if mccs[i] <= 0 else (float(mccs[i]), float(cuts[i]))

def best_threshold_two_sided(y_true, p_pos):
    """Max MCC over both orientations. Returns (mcc, thr, orient) with orient +1 (p>=thr) or -1 ((1-p)>=thr)."""
    m_pos, t_pos = best_threshold_mcc(y_true, p_pos)
    m_neg, t_neg = best_threshold_mcc(y_true, 1.0 - np.asarray(p_pos))
    return (m_pos, t_pos, 1) if m_pos >= m_neg else (m_neg, t_neg, -1)

def apply_oriented(p_pos, thr, orient):
    p = np.asarray(p_pos)
    return ((p if orient == 1 else 1.0 - p) >= thr).astype(int)

def mcc_of_pred(y_true, pred):
    return matthews_corrcoef(np.asarray(y_true), np.asarray(pred))

# ---------- acquisition rules: return integer positions into the pool ----------
def acquire_uniform(n_pool, k, seed):
    return np.random.default_rng(seed).choice(n_pool, size=min(k, n_pool), replace=False)

def acquire_uncertainty(p_pool, k):
    p = np.clip(np.asarray(p_pool), 1e-9, 1 - 1e-9)
    ent = -(p * np.log(p) + (1 - p) * np.log(1 - p))
    return np.argsort(-ent, kind='mergesort')[:k]

def acquire_diversity(X_pool, k, seed):
    """k-means with k clusters on standardised features; per cluster, the member nearest its centroid.
    Distances are computed to each row's own centroid only (O(n*features)), never as an n x k matrix."""
    Xs = StandardScaler().fit_transform(np.asarray(X_pool, dtype=np.float64))
    k = min(k, len(Xs))
    km = MiniBatchKMeans(n_clusters=k, random_state=seed, batch_size=4096, n_init=1, max_iter=50).fit(Xs)
    labels = km.labels_
    own = np.einsum('ij,ij->i', Xs - km.cluster_centers_[labels], Xs - km.cluster_centers_[labels])
    df = pd.DataFrame({'lab': labels, 'd': own})
    chosen = df.groupby('lab').d.idxmin().values.astype(int)
    if len(chosen) < k:
        rest = np.setdiff1d(np.arange(len(Xs)), chosen)
        chosen = np.concatenate([chosen, rest[np.argsort(own[rest])[:k - len(chosen)]]])
    return chosen

def acquire_hybrid(X_pool, p_pool, k, seed, factor=5):
    cand = acquire_uncertainty(p_pool, min(len(p_pool), factor * k))
    sub = acquire_diversity(np.asarray(X_pool)[cand], k, seed)
    return cand[sub]

# ---------- cross-validated buffer-only estimate on the buffer itself ----------
def cv_estimate(make_model_fn, Xb, yb, seed, n_splits=3):
    """Mean MCC over stratified folds; returns 0.0 when a class has fewer than n_splits rows."""
    yb = np.asarray(yb)
    if len(np.unique(yb)) < 2 or np.bincount(yb).min() < n_splits:
        return 0.0
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    out = []
    for tr, te in skf.split(Xb, yb):
        if len(np.unique(yb[tr])) < 2:
            out.append(0.0); continue
        mdl = make_model_fn(len(tr)); mdl.fit(Xb.iloc[tr], yb[tr])
        out.append(mcc_of_pred(yb[te], (mdl.predict_proba(Xb.iloc[te])[:, 1] >= 0.5).astype(int)))
    return float(np.mean(out))

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import matthews_corrcoef
import lightgbm as lgb

# ---------- fine-tuning: keep the source model, adapt it on the buffer ----------
def finetune(model, name, Xb, yb, seed, add_trees=100, ft_epochs=30):
    """Family-specific continuation of a fitted source model on the buffer.
    RF: warm_start adds `add_trees` grown on the buffer, source trees retained.
    LightGBM: boosting continues from the source booster via init_model.
    MLP: SGD continues from the source weights, with the buffer transformed by the
    SOURCE scaler; refitting the pipeline would refit the scaler on the buffer and
    apply the source weights to differently-scaled inputs.
    A single-class buffer offers nothing to adapt to, so the source model is returned."""
    yb = np.asarray(yb)
    if len(np.unique(yb)) < 2:
        return model
    if name == 'rf':
        model.set_params(warm_start=True, n_estimators=model.n_estimators + add_trees)
        model.fit(Xb, yb)
        return model
    if name == 'lgbm':
        m2 = lgb.LGBMClassifier(n_estimators=add_trees, random_state=seed, n_jobs=-1, verbosity=-1)
        m2.fit(Xb, yb, init_model=model.booster_)
        return m2
    scaler = model.named_steps['scaler']
    clf = model.named_steps['clf']
    # sklearn leaves best_loss_ unset when the source fit used early stopping; the
    # non-early-stopping update path dereferences it, so it is restored here.
    if getattr(clf, 'best_loss_', None) is None:
        curve = getattr(clf, 'loss_curve_', None)
        clf.best_loss_ = float(curve[-1]) if curve else np.inf
    clf._no_improvement_count = 0
    clf.set_params(warm_start=True, max_iter=ft_epochs, early_stopping=False)
    clf.fit(scaler.transform(Xb), yb)
    return model


# ---------- importance weighting: reweight source rows towards the target ----------
def importance_weights(Xs, Xb, seed, clip=(0.05, 20.0), n_src=20000):
    """Domain discriminator p(target|x); source weight = p/(1-p).
    The discriminator is trained on a class-balanced sample and with balanced class
    weights, because an unbalanced source/buffer ratio otherwise drives every source
    probability to zero and collapses the weights onto the clip floor."""
    rng = np.random.default_rng(seed)
    n = min(n_src, len(Xs), max(len(Xb), 200))
    idx = rng.choice(len(Xs), size=min(n_src, len(Xs)), replace=False)
    src_d = np.asarray(Xs)[rng.choice(len(Xs), size=n, replace=False)]
    tgt_d = np.asarray(Xb)
    if len(tgt_d) < n:
        tgt_d = tgt_d[rng.choice(len(tgt_d), size=n, replace=True)]
    Xd = np.vstack([src_d, tgt_d])
    yd = np.r_[np.zeros(len(src_d)), np.ones(len(tgt_d))]
    mu, sd = Xd.mean(0), Xd.std(0) + 1e-9
    lr = LogisticRegression(max_iter=300, C=1.0, class_weight='balanced')
    lr.fit((Xd - mu) / sd, yd)
    p = lr.predict_proba((np.asarray(Xs) - mu) / sd)[:, 1]
    w = p / np.clip(1 - p, 1e-6, None)
    w = np.clip(w, *clip)
    return w / w.mean()


# ---------- selector estimator that does not collapse on tiny buffers ----------
def cv_estimate_v2(make_model_fn, Xb, yb, seed):
    """Stratified CV with folds adapted to the rarest class; falls back to a
    single stratified holdout when the minority class has 2 rows, and to the
    buffer-resubstitution estimate when it has 1. Never returns 0 by default."""
    yb = np.asarray(yb)
    if len(np.unique(yb)) < 2:
        return 0.0, 'single_class'
    nmin = int(np.bincount(yb).min())
    if nmin >= 3:
        k = min(3, nmin)
        skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
        out = []
        for tr, te in skf.split(Xb, yb):
            if len(np.unique(yb[tr])) < 2:
                continue
            m = make_model_fn(len(tr)); m.fit(Xb.iloc[tr], yb[tr])
            out.append(matthews_corrcoef(yb[te], (m.predict_proba(Xb.iloc[te])[:, 1] >= 0.5).astype(int)))
        return (float(np.mean(out)), f'cv{k}') if out else (0.0, 'cv_empty')
    if nmin == 2:
        skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=seed)
        out = []
        for tr, te in skf.split(Xb, yb):
            if len(np.unique(yb[tr])) < 2:
                continue
            m = make_model_fn(len(tr)); m.fit(Xb.iloc[tr], yb[tr])
            out.append(matthews_corrcoef(yb[te], (m.predict_proba(Xb.iloc[te])[:, 1] >= 0.5).astype(int)))
        if out and max(out) > 0:
            return float(np.mean(out)), 'holdout2'
        m = make_model_fn(len(Xb)); m.fit(Xb, yb)
        return float(matthews_corrcoef(yb, (m.predict_proba(Xb)[:, 1] >= 0.5).astype(int))), 'resub2'
    m = make_model_fn(len(Xb)); m.fit(Xb, yb)
    return float(matthews_corrcoef(yb, (m.predict_proba(Xb)[:, 1] >= 0.5).astype(int))), 'resub'

# ---------- iterative active learning: the retrained model drives the queries ----------
def entropy(p):
    p = np.clip(np.asarray(p), 1e-9, 1 - 1e-9)
    return -(p * np.log(p) + (1 - p) * np.log(1 - p))

def al_iterative(make_model_fn, pool, features, clean_fn, k_total, rounds, seed, stratify_col='Attack'):
    """Round 1 is a stratified random seed set; later rounds query the highest-entropy
    pool rows under the model retrained on what has been labelled so far."""
    per = max(1, k_total // rounds)
    rng_state = seed
    first = pool.groupby(stratify_col, group_keys=False).apply(
        lambda g: g.sample(n=max(1, int(round(len(g) * per / len(pool)))), random_state=rng_state))
    chosen = list(first.index[:per])
    for _ in range(rounds - 1):
        lab = pool.loc[chosen]
        Xl, med = clean_fn(lab, features)
        yl = lab['Label'].values
        if len(np.unique(yl)) < 2:
            rest = pool.index.difference(chosen)
            chosen += list(pd.Index(rest)[:per]); continue
        m = make_model_fn(len(Xl)); m.fit(Xl, yl)
        rest = pool.index.difference(chosen)
        Xr, _ = clean_fn(pool.loc[rest], features, medians=med)
        e = entropy(m.predict_proba(Xr)[:, 1])
        chosen += list(pd.Index(rest)[np.argsort(-e)[:per]])
    return pool.loc[chosen[:k_total]]

# ---------- INSOMNIA-style: pseudo-labelled pool plus uncertainty-queried labels ----------
def insomnia_buffer(source_model, pool, features, clean_fn, med_s, k, seed, conf=0.9, cap=50000):
    """Approximates INSOMNIA: the frozen source model pseudo-labels confident pool rows,
    and the analyst budget is spent on the rows it is least certain about."""
    Xp, _ = clean_fn(pool, features, medians=med_s)
    p = source_model.predict_proba(Xp)[:, 1]
    q = pool.index[np.argsort(-entropy(p))[:k]]
    conf_mask = (p >= conf) | (p <= 1 - conf)
    conf_idx = pool.index[conf_mask].difference(q)
    if len(conf_idx) > cap:
        conf_idx = pd.Index(np.random.default_rng(seed).choice(conf_idx, cap, replace=False))
    pseudo = pool.loc[conf_idx].copy()
    pseudo['Label'] = (p[pool.index.get_indexer(conf_idx)] >= 0.5).astype(int)
    return pool.loc[q], pseudo

In [ ]:
from sklearn.model_selection import train_test_split

def record(row):
    r = {c: row.get(c, np.nan) for c in COLS}
    pd.DataFrame([r], columns=COLS).to_csv(V4_CSV, mode='a', index=False, header=not os.path.exists(V4_CSV))

def key(seed, src, tgt, m, b, s):
    return (str(seed), src, tgt, m, f'{float(b):.6g}', s)

done = set()
if os.path.exists(V4_CSV):
    prev = pd.read_csv(V4_CSV)
    done = set(key(r.seed, r.source, r.target, r.model, r.budget, r.strategy) for r in prev.itertuples())
    print(f'resume: {len(done)} rows already recorded')

def is_done(*k):
    return key(*k) in done

def mark(seed, src, tgt, m, b, s, metrics, n_train, **extra):
    record(dict(seed=seed, source=src, target=tgt, model=m, budget=b, strategy=s,
                n_train=n_train, **metrics, **extra))
    done.add(key(seed, src, tgt, m, b, s))
    print(f"  s{seed} {src}->{tgt} {m} b={b} {s}: MCC={metrics['mcc']:.3f} FPR={metrics['fp_rate']:.3f}")

def fit_eval(mname, seed, Xtr, ytr, Xev, yev, weights=None):
    ytr = np.asarray(ytr)
    if len(np.unique(ytr)) < 2:
        m = all_metrics(yev, np.full(len(yev), float(ytr[0]))); m['mcc'] = 0.0
        return m, 0
    model = make_model(mname, seed, len(Xtr))
    t0 = time.time()
    if weights is None:
        model.fit(Xtr, ytr)
    elif mname == 'mlp':
        model.fit(Xtr, ytr)
    else:
        model.fit(Xtr, ytr, sample_weight=weights)
    m = all_metrics(yev, model.predict_proba(Xev)[:, 1])
    fs = round(time.time() - t0); del model; gc.collect()
    return m, fs

for seed in CFG['seeds']:
    parts = {}
    for tag, d in DATASETS.items():
        tr, te = train_test_split(d, test_size=CFG['test_size'], stratify=d['Attack'], random_state=seed)
        tr = tr.reset_index(drop=True)
        parts[tag] = dict(train_full=tr, train=stratified_cap(tr, CFG['train_cap'], seed),
                          eval=stratified_cap(te, CFG['eval_cap'], seed),
                          pool=stratified_cap(tr, CFG['pool_cap'], seed).reset_index(drop=True))

    for src in CFG['corpora']:
        others = [t for t in CFG['corpora'] if t != src]
        Xs, med_s = clean_X(parts[src]['train'], FEATURES)
        ys = parts[src]['train']['Label'].values
        for mname in MODELS:
            todo = []
            for tgt in others:
                for b in CFG['src_budgets']:
                    todo += [not is_done(seed, src, tgt, mname, b, 'finetune')]
                    if seed == CFG['ref_seed']:
                        todo += [not is_done(seed, src, tgt, mname, b, 'iw_augment')]
                for b in CFG['acq_budgets']:
                    todo += [not is_done(seed, src, tgt, mname, b, 'al_iterative'),
                             not is_done(seed, src, tgt, mname, b, 'insomnia')]
                for b in CFG['budgets']:
                    todo += [not is_done(seed, src, tgt, mname, b, 'selector_v2')]
            if not any(todo):
                continue
            src_model = make_model(mname, seed, len(Xs))
            t0 = time.time(); src_model.fit(Xs, ys)
            print(f'seed {seed} | source fit {mname} on {src}: {time.time()-t0:.0f}s')

            for tgt in others:
                ev = parts[tgt]['eval']; yev = ev['Label'].values
                Xev_s, _ = clean_X(ev, FEATURES, medians=med_s)
                p_ev = src_model.predict_proba(Xev_s)[:, 1]
                pool = parts[tgt]['pool']

                for b in CFG['budgets']:
                    need_sel = not is_done(seed, src, tgt, mname, b, 'selector_v2')
                    need_ft  = b in CFG['src_budgets'] and not is_done(seed, src, tgt, mname, b, 'finetune')
                    need_iw  = (seed == CFG['ref_seed'] and b in CFG['src_budgets']
                                and not is_done(seed, src, tgt, mname, b, 'iw_augment'))
                    if not (need_sel or need_ft or need_iw):
                        continue
                    buf = stratified_frac(parts[tgt]['train_full'], b, seed)
                    ybuf = buf['Label'].values
                    nben = int((ybuf == 0).sum())
                    Xb_s, _ = clean_X(buf, FEATURES, medians=med_s)

                    if need_sel:
                        pb = src_model.predict_proba(Xb_s)[:, 1]
                        bm, bt, bo = best_threshold_two_sided(ybuf, pb)
                        Xbf, _ = clean_X(buf, FEATURES)
                        cv, mode = cv_estimate_v2(lambda n: make_model(mname, seed, n), Xbf, ybuf, seed)
                        m = all_metrics(yev, p_ev); m['mcc'] = np.nan
                        mark(seed, src, tgt, mname, b, 'selector_v2', m, len(buf),
                             buffer_est=bm, cv_est=cv, cv_mode=mode, n_benign_buf=nben)
                        del Xbf

                    if need_ft:
                        import copy
                        ftm = finetune(copy.deepcopy(src_model), mname, Xb_s, ybuf, seed,
                                       add_trees=CFG['ft_add_trees'], ft_epochs=CFG['ft_epochs'])
                        t1 = time.time()
                        m = all_metrics(yev, ftm.predict_proba(Xev_s)[:, 1])
                        mark(seed, src, tgt, mname, b, 'finetune', m, len(Xs) + len(buf),
                             n_benign_buf=nben, fit_s=round(time.time() - t1))
                        del ftm; gc.collect()

                    if need_iw:
                        w = importance_weights(Xs.values, Xb_s.values, seed)
                        both = pd.concat([parts[src]['train'], buf], ignore_index=True)
                        Xa, ma = clean_X(both, FEATURES)
                        Xev_a, _ = clean_X(ev, FEATURES, medians=ma)
                        wa = np.r_[w, np.ones(len(buf))]
                        m, fs = fit_eval(mname, seed, Xa, both['Label'].values, Xev_a, yev, weights=wa)
                        mark(seed, src, tgt, mname, b, 'iw_augment', m, len(both),
                             n_benign_buf=nben, fit_s=fs)
                        del both, Xa, Xev_a; gc.collect()
                    del Xb_s

                for b in CFG['acq_budgets']:
                    k = max(1, int(round(len(parts[tgt]['train_full']) * b)))
                    if not is_done(seed, src, tgt, mname, b, 'al_iterative'):
                        bufa = al_iterative(lambda n: make_model(mname, seed, n), pool, FEATURES,
                                            clean_X, k, CFG['al_rounds'], seed)
                        Xb2, mb = clean_X(bufa, FEATURES); Xev_b, _ = clean_X(ev, FEATURES, medians=mb)
                        m, fs = fit_eval(mname, seed, Xb2, bufa['Label'].values, Xev_b, yev)
                        mark(seed, src, tgt, mname, b, 'al_iterative', m, len(bufa),
                             n_benign_buf=int((bufa['Label'] == 0).sum()), fit_s=fs)
                        del Xb2, Xev_b; gc.collect()
                    if not is_done(seed, src, tgt, mname, b, 'insomnia'):
                        q, pseudo = insomnia_buffer(src_model, pool, FEATURES, clean_X, med_s, k, seed,
                                                    conf=CFG['pseudo_conf'], cap=CFG['pseudo_cap'])
                        trn = pd.concat([q, pseudo], ignore_index=True)
                        Xt, mt = clean_X(trn, FEATURES); Xev_i, _ = clean_X(ev, FEATURES, medians=mt)
                        m, fs = fit_eval(mname, seed, Xt, trn['Label'].values, Xev_i, yev)
                        mark(seed, src, tgt, mname, b, 'insomnia', m, len(trn),
                             n_pseudo=len(pseudo), n_benign_buf=int((q['Label'] == 0).sum()), fit_s=fs)
                        del trn, Xt, Xev_i; gc.collect()
                del Xev_s, p_ev; gc.collect()
            del src_model; gc.collect()
        del Xs; gc.collect()

print('rows recorded:', len(done))

In [ ]:
from scipy.stats import wilcoxon

v1 = pd.read_csv(f'{RESULT}/fc_results.csv').drop_duplicates(['seed','source','target','model','budget','strategy'])
v3 = pd.read_csv(f'{RESULT}/fc_results_v3.csv').drop_duplicates(['seed','source','target','model','budget','strategy'])
v4 = pd.read_csv(V4_CSV).drop_duplicates(['seed','source','target','model','budget','strategy'])
K = ['seed','source','target','model','budget']

print('=== SOURCE USE: matched cells at the reference seed ===')
bo = v1[v1.strategy=='buffer_only'][K+['mcc','fp_rate']].rename(columns={'mcc':'buffer_only','fp_rate':'fpr_bo'})
au = v1[v1.strategy=='augment'][K+['mcc']].rename(columns={'mcc':'augment'})
ft = v4[v4.strategy=='finetune'][K+['mcc']].rename(columns={'mcc':'finetune'})
iw = v4[v4.strategy=='iw_augment'][K+['mcc']].rename(columns={'mcc':'iw_augment'})
S = bo.merge(au,on=K).merge(ft,on=K).merge(iw,on=K)
print(S.groupby('budget')[['buffer_only','augment','iw_augment','finetune']].agg(['mean','size']).round(3).to_string())
print()
for c in ['augment','iw_augment','finetune']:
    d = S.buffer_only - S[c]
    pair = S.groupby(['source','target'])[['buffer_only',c]].mean()
    w_cell = wilcoxon(S.buffer_only, S[c]).pvalue
    w_pair = wilcoxon(pair.buffer_only, pair[c]).pvalue
    print(f'buffer_only vs {c:11s}: mean diff {d.mean():+.4f} | wins {int((d>0).sum())}/{len(d)} '
          f'| cell-level p={w_cell:.2e} | PAIR-level p={w_pair:.3f} (n={len(pair)})')

print('\n=== BASELINE COMPARISON: acquisition rules, all seeds ===')
acq = v3[v3.strategy.str.startswith('acq_')].copy(); acq['rule'] = acq.strategy.str.replace('acq_','')
shared = acq[acq.source=='any'].groupby(['seed','target','model','budget','rule']).mcc.mean().reset_index()
srcdep = acq[acq.source!='any'].groupby(['seed','target','model','budget','rule']).mcc.mean().reset_index()
new = v4[v4.strategy.isin(['al_iterative','insomnia'])].groupby(['seed','target','model','budget','strategy']).mcc.mean().reset_index().rename(columns={'strategy':'rule'})
base = v1[(v1.strategy=='buffer_only')&(v1.budget.isin(CFG['acq_budgets']))].groupby(['seed','target','model','budget']).mcc.mean().rename('strat_random').reset_index()
A = pd.concat([shared,srcdep,new]).merge(base,on=['seed','target','model','budget'])
A['gain'] = A.mcc - A.strat_random
rows=[]
for (b,r),g in A.groupby(['budget','rule']):
    pair = g.groupby('target')[['mcc','strat_random']].mean()
    rows.append(dict(budget=b, rule=r, n=len(g), rule_mcc=g.mcc.mean(), baseline=g.strat_random.mean(),
                     gain=g.gain.mean(), frac_better=(g.gain>0).mean(),
                     p_cell=wilcoxon(g.mcc,g.strat_random).pvalue,
                     p_target=wilcoxon(pair.mcc,pair.strat_random).pvalue if len(pair)>2 else np.nan))
AT = pd.DataFrame(rows).round(4)
print(AT.to_string(index=False))
AT.to_csv(f'{RESULT}/fc_acquisition_v2.csv', index=False)
print('\nper-target gain of each rule over stratified random:')
print(A.groupby(['budget','rule','target']).gain.mean().round(3).unstack().to_string())

print('\n=== SELECTOR v2 ===')
sel = v4[v4.strategy=='selector_v2'][K+['buffer_est','cv_est','cv_mode']]
rt = v3[v3.strategy=='rethreshold_2s'][K+['mcc']].rename(columns={'mcc':'rt'})
S2 = sel.merge(rt,on=K).merge(v1[v1.strategy=='buffer_only'][K+['mcc']].rename(columns={'mcc':'bo'}),on=K)
S2['chose_rt'] = (S2.buffer_est > S2.cv_est).astype(int)
S2['realised'] = np.where(S2.chose_rt==1, S2.rt, S2.bo)
S2['oracle'] = np.maximum(S2.rt, S2.bo)
S2['regret'] = S2.oracle - S2.realised
S2['always_retrain'] = S2.oracle - S2.bo
S2['correct'] = (S2.chose_rt==1) == (S2.rt >= S2.bo)
print(S2.groupby('budget').agg(chose_rt=('chose_rt','mean'), regret=('regret','mean'),
      always_retrain=('always_retrain','mean'), acc=('correct','mean')).round(4).to_string())
print('\nestimator mode used, by budget:')
print(S2.groupby(['budget','cv_mode']).size().unstack(fill_value=0).to_string())
print('\nrestricted to cells with a genuine cross-validated estimate:')
g = S2[S2.cv_mode.str.startswith('cv')]
print(g.groupby('budget').agg(n=('regret','size'), regret=('regret','mean'),
      always_retrain=('always_retrain','mean'), acc=('correct','mean')).round(4).to_string())
S2.to_csv(f'{RESULT}/fc_selector_v2.csv', index=False)

print('\n=== FALSE-POSITIVE BURDEN of every strategy ===')
allr = pd.concat([v1.assign(src_file='v1'), v3.assign(src_file='v3'), v4.assign(src_file='v4')])
fpr = allr[allr.strategy.isin(['buffer_only','augment','iw_augment','finetune','al_iterative','insomnia',
                               'acq_diversity','acq_uncertainty','calibrate','rethreshold_2s'])]
print(fpr.groupby(['strategy','budget']).agg(mcc=('mcc','mean'), fpr=('fp_rate','mean'),
      fpr_med=('fp_rate','median'), fpr_max=('fp_rate','max')).round(4).to_string())
fpr.groupby(['strategy','budget']).agg(mcc=('mcc','mean'), fpr_mean=('fp_rate','mean'),
    fpr_med=('fp_rate','median'), fpr_max=('fp_rate','max')).round(4).reset_index().to_csv(f'{RESULT}/fc_fpr.csv', index=False)
print('\nbuffer_only FP-rate by target and budget:')
print(v1[v1.strategy=='buffer_only'].groupby(['budget','target']).fp_rate.mean().round(4).unstack().to_string())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
os.chdir('/content/drive/MyDrive/drift-conference')

r = subprocess.run(["python", "tools/commit_cell.py",
  "13: source-use strategies (finetune, importance-weighted augment), INSOMNIA-style and iterative-AL baselines, corrected selector"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)